# 03.00 · Ejecutar el paso 3: insumo de Evidencia + Recomendaciones

Corre en orden el `03.01` a `03.04`: para los IDOs PPI se genera su insumo en Word en
`data/processed/3. Insumo Evid+Recom/`. Los IDOs que no son PPI no pasan al paso 3.

El paso 3 no ejecuta nada del paso 2: solo lee su salida final, `idos_concepto.csv`. Por eso,
**primero se corre el `02.00`**, que procesa los Word nuevos y donde se aprueba o edita el
concepto de priorización; después se corre este notebook. Un IDO que todavía no pasó por el
`02.00` no aparece en el paso 3.

Funciona como el `02.00`: cada notebook corre en su propio kernel con el Python de `.venv`, los
notebooks originales no se modifican y el flujo se detiene en el primer fallo. Solo los IDOs
nuevos o modificados llaman a Azure; el resto sale de las respuestas guardadas. Al final se
agrega un resumen a `reports/3. Insumo Evid+Recom_historico_ejecuciones.xlsx`.

**Antes de ejecutar, se cierran en Positron los notebooks del paso 3.**

### Parámetros

`notebooks_a_ejecutar` lleva las rutas dentro de `notebooks/`, en el orden del flujo.
`timeout_por_celda` está en segundos: 3.600 permite hasta una hora por celda.

In [1]:
notebooks_a_ejecutar = [
    "3. Insumo Evid+Recom/03.01_datos_ido.ipynb",
    "3. Insumo Evid+Recom/03.02_seccion1_evidencia.ipynb",
    "3. Insumo Evid+Recom/03.03_seccion2_implicaciones.ipynb",
    "3. Insumo Evid+Recom/03.04_word_insumo.ipynb",
]
timeout_por_celda = 3600

## 1. LOAD LIBRARIES

Las mismas del `02.00`: nbformat lee los notebooks, nbclient los ejecuta y jupyter_client fija
el Python del proyecto. Las rutas se toman de `caf_input_cci_preparatoria/config.py`.

In [2]:
from datetime import datetime
import json
from pathlib import Path
import re
import sys
import tempfile
from time import perf_counter

from dotenv import dotenv_values
from IPython.display import display
from jupyter_client import KernelManager
from jupyter_client.kernelspec import KernelSpecManager
from nbclient import NotebookClient
import nbformat
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))  # para importar el paquete caf_input_cci_preparatoria

from caf_input_cci_preparatoria import config

## 2. LOAD DATA

Los valores del `.env` se leen sin imprimirlos, solo para ocultarlos si aparecieran en un
error. El `kernel.json` temporal le indica a Jupyter que use `.venv/bin/python`, como en el
`02.00`.

In [3]:
directorio_notebooks = ROOT / "notebooks"
python_proyecto = ROOT / ".venv" / "bin" / "python"
assert python_proyecto.is_file(), "No se encuentra el intérprete del proyecto en .venv/bin/python"

archivo_historico = config.REPORTS / "3. Insumo Evid+Recom_historico_ejecuciones.xlsx"
archivo_idos_ppi = config.INTERIM / "3. Insumo Evid+Recom" / "03.01_datos_ido" / "idos_insumo.csv"
valores_privados = [valor for valor in dotenv_values(ROOT / ".env").values() if valor]

In [4]:
nombre_kernel = "caf-proyecto"
carpeta_temporal = tempfile.TemporaryDirectory()
directorio_kernel = Path(carpeta_temporal.name) / nombre_kernel
directorio_kernel.mkdir()
especificacion_kernel = {
    "argv": [str(python_proyecto), "-m", "ipykernel_launcher", "-f", "{connection_file}"],
    "display_name": "Python del proyecto CAF",
    "language": "python",
}
(directorio_kernel / "kernel.json").write_text(json.dumps(especificacion_kernel), encoding="utf-8")
gestor_especificaciones = KernelSpecManager(
    kernel_dirs=[carpeta_temporal.name], ensure_native_kernel=False
)

## 3. PROCESS DATA

Cada notebook se ejecuta en un kernel nuevo con su propia carpeta como directorio de trabajo.
Del texto de salida se toma el uso de Azure: la línea `Uso de Azure: …` de cada notebook.

En la fila del `03.01` se anotan los IDOs PPI y los nuevos (huella que no estaba en
`idos_insumo.csv` antes de la corrida); en la del `03.04`, los Word generados.

In [5]:
def ocultar(texto):
    for valor in valores_privados:
        texto = texto.replace(valor, "[OCULTO]")
    return texto


def texto_salidas(notebook):
    return "".join(
        salida.get("text", "") for celda in notebook.cells for salida in celda.get("outputs", [])
    )


def uso_de_azure(notebook):
    patron = (
        r"Uso de Azure: ([\d,]+) llamadas; "
        r"([\d,]+) tokens de entrada; ([\d,]+) tokens de salida"
    )
    encontrado = re.search(patron, texto_salidas(notebook))
    if encontrado is None:
        return {}
    valores = [int(numero.replace(",", "")) for numero in encontrado.groups()]
    return dict(zip(["Llamadas Azure", "Tokens entrada", "Tokens salida"], valores))


def leer_idos_ppi():
    if not archivo_idos_ppi.exists():
        return pd.DataFrame(columns=["archivo", "huella"])
    return pd.read_csv(
        archivo_idos_ppi, encoding="utf-8-sig", usecols=["archivo", "huella"], dtype=str
    )


def celda_con_error(notebook):
    for numero, celda in enumerate(notebook.cells, start=1):
        if any(salida.output_type == "error" for salida in celda.get("outputs", [])):
            return numero
    return None

In [6]:
fecha_ejecucion = datetime.now().strftime("%Y-%m-%d %H:%M")
idos_ppi_antes = leer_idos_ppi()
resultados = []
for ruta in notebooks_a_ejecutar:
    nombre = Path(ruta).name
    carpeta = (directorio_notebooks / ruta).parent
    print(f"Ejecutando {ruta}...")
    notebook = nbformat.read(directorio_notebooks / ruta, as_version=4)
    for celda in notebook.cells:
        if celda.cell_type == "code":
            celda.outputs, celda.execution_count = [], None
    cliente = NotebookClient(
        notebook,
        km=KernelManager(kernel_name=nombre_kernel, kernel_spec_manager=gestor_especificaciones),
        timeout=timeout_por_celda,
        resources={"metadata": {"path": str(carpeta)}},
    )
    inicio = perf_counter()
    error = ""
    try:
        cliente.execute()
    except Exception as fallo:  # noqa: BLE001 - el fallo queda en el resumen y detiene el flujo
        tipo = getattr(fallo, "ename", type(fallo).__name__)
        error = ocultar(f"{tipo}: {getattr(fallo, 'evalue', fallo)}")
    resultados.append(
        {
            "Fecha y hora": fecha_ejecucion,
            "Notebook": nombre,
            "Estado": "Falló" if error else "Completado",
            "Duración (s)": round(perf_counter() - inicio, 1),
            **uso_de_azure(notebook),
            "Celda": celda_con_error(notebook) if error else None,
            "Error": error,
        }
    )
    if nombre.startswith("03.01") and not error:
        idos_ppi = leer_idos_ppi()
        nuevos = idos_ppi.loc[~idos_ppi["huella"].isin(idos_ppi_antes["huella"]), "archivo"]
        resultados[-1].update(
            {
                "IDOs PPI": len(idos_ppi),
                "IDOs PPI nuevos": len(nuevos),
                "Archivos nuevos": "; ".join(nuevos),
            }
        )
    if nombre.startswith("03.04") and not error:
        resultados[-1]["Word generados"] = texto_salidas(notebook).count(".docx")
    if error:
        print(f"Falló {nombre}: {error}")
        break
carpeta_temporal.cleanup()

Ejecutando 3. Insumo Evid+Recom/03.01_datos_ido.ipynb...
Ejecutando 3. Insumo Evid+Recom/03.02_seccion1_evidencia.ipynb...
Ejecutando 3. Insumo Evid+Recom/03.03_seccion2_implicaciones.ipynb...
Ejecutando 3. Insumo Evid+Recom/03.04_word_insumo.ipynb...


## 4. SAVE DATA

Se muestra el resumen de esta corrida, una fila por notebook ejecutado, con el total de
llamadas y tokens de Azure. Las mismas filas se agregan
a `reports/3. Insumo Evid+Recom_historico_ejecuciones.xlsx`, que se crea la primera vez.

In [7]:
columnas = [
    "Fecha y hora",
    "Notebook",
    "Estado",
    "Duración (s)",
    "IDOs PPI",
    "IDOs PPI nuevos",
    "Archivos nuevos",
    "Word generados",
    "Llamadas Azure",
    "Tokens entrada",
    "Tokens salida",
    "Error",
    "Celda",
]
columnas_enteras = [
    "IDOs PPI",
    "IDOs PPI nuevos",
    "Word generados",
    "Llamadas Azure",
    "Tokens entrada",
    "Tokens salida",
    "Celda",
]

resumen = pd.DataFrame(resultados).reindex(columns=columnas)
resumen[columnas_enteras] = resumen[columnas_enteras].astype("Int64")  # enteros, no 3.0
with pd.option_context("display.max_colwidth", None):
    display(resumen)
totales = resumen[["Llamadas Azure", "Tokens entrada", "Tokens salida"]].sum()
print(
    f"Azure en esta corrida: {totales['Llamadas Azure']} llamadas; "
    f"{totales['Tokens entrada']:,} tokens de entrada; "
    f"{totales['Tokens salida']:,} tokens de salida"
)

,Fecha y hora,Notebook,Estado,Duración (s),IDOs PPI,IDOs PPI nuevos,Archivos nuevos,Word generados,Llamadas Azure,Tokens entrada,Tokens salida,Error,Celda
0,2026-11-13 16:30,03.01_datos_ido.ipynb,Completado,1.0,13,2,CCI-IDO-Consolidación Plan INADEH v2.docx; FR-228 guanay actualizado.docx,<NA>,<NA>,<NA>,<NA>,,<NA>
1,2026-11-13 16:30,03.02_seccion1_evidencia.ipynb,Completado,79.8,<NA>,<NA>,NaN,<NA>,2,19558,4041,,<NA>
2,2026-11-13 16:30,03.03_seccion2_implicaciones.ipynb,Completado,68.1,<NA>,<NA>,NaN,<NA>,2,25116,3256,,<NA>
3,2026-11-13 16:30,03.04_word_insumo.ipynb,Completado,1.0,<NA>,<NA>,NaN,13,<NA>,<NA>,<NA>,,<NA>


Azure en esta corrida: 4 llamadas; 44,674 tokens de entrada; 7,297 tokens de salida


In [8]:
anteriores = pd.read_excel(archivo_historico) if archivo_historico.exists() else None
historico = pd.concat([anteriores, resumen], ignore_index=True).reindex(columns=columnas)
historico[columnas_enteras] = historico[columnas_enteras].astype("Int64")
archivo_historico.parent.mkdir(parents=True, exist_ok=True)
historico.to_excel(archivo_historico, index=False)
print(f"{len(resumen)} filas agregadas a {archivo_historico.relative_to(ROOT)}")

4 filas agregadas a reports/3. Insumo Evid+Recom_historico_ejecuciones.xlsx
